In [6]:
%pip install statsmodels

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Source Han Sans CN']


Note: you may need to restart the kernel to use updated packages.


In [7]:
# 1. 导入数据
print("=" * 50)
print("导入数据")
print("=" * 50)

# 读取训练集数据
df = pd.read_csv('rent_price_train_dataset.csv')

print(f"训练集数据形状: {df.shape}")
print(f"变量数量: {len(df.columns)}")
print(f"记录数量: {len(df)}")

# 检查数据基本信息
print("\n训练集基本信息:")
print(df.info())

# 显示前几行数据，确认数据正确加载
print("\n前5行数据:")
print(df.head())

# 检查列名，确认所有需要的变量都存在
print("\n数据集列名:")
print(df.columns.tolist())

导入数据
训练集数据形状: (98899, 82)
变量数量: 82
记录数量: 98899

训练集基本信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 82 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   decoration              98899 non-null  int64  
 1   area                    98899 non-null  float64
 2   室                       98899 non-null  int64  
 3   厅                       98899 non-null  int64  
 4   卫                       98899 non-null  int64  
 5   总楼层                     98899 non-null  float64
 6   准确楼层                    98899 non-null  float64
 7   south_dummy             98899 non-null  float64
 8   north_south_dummy       98899 non-null  float64
 9   pay_annual              98899 non-null  float64
 10  pay_bi_monthly          98899 non-null  float64
 11  pay_monthly             98899 non-null  float64
 12  pay_quarterly           98899 non-null  float64
 13  pay_semi_annual         98899 non-

In [17]:
# 2.1 OLS 模型建模
# OLS 模型建模
print("\n" + "=" * 60)
print("OLS 模型建模")
print("=" * 60)

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 读取训练集数据
print("读取训练集数据...")
df = pd.read_csv('rent_price_train_dataset.csv')
print(f"训练集形状: {df.shape}")

# 准备特征变量和目标变量
print("\n准备特征变量和目标变量...")

# 选择特征 - 排除目标变量和ID等
feature_cols = [col for col in df.columns if col not in ['Price', 'lnPrice', 'ID'] 
                and df[col].dtype in ['int64', 'float64']]

X = df[feature_cols]
y_price = df['Price']  # 原始价格
y_lnprice = df['lnPrice']  # 对数价格

print(f"特征数量: {len(feature_cols)}")
print(f"样本数量: {len(X)}")

# 处理缺失值
print("\n处理缺失值...")
for col in X.columns:
    if X[col].isnull().sum() > 0:
        null_count = X[col].isnull().sum()
        if X[col].dtype in ['float64', 'int64']:
            median_val = X[col].median()
            X[col] = X[col].fillna(median_val)
            print(f"  - {col}: {null_count} 个缺失值已用中位数填充")
        else:
            mode_val = X[col].mode()[0] if not X[col].mode().empty else 0
            X[col] = X[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已用众数填充")

# 划分训练集和测试集
print("\n划分训练集和测试集...")
X_train, X_test, y_train_price, y_test_price = train_test_split(
    X, y_price, test_size=0.2, random_state=42
)
_, _, y_train_lnprice, y_test_lnprice = train_test_split(
    X, y_lnprice, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 标准化特征
print("\n标准化特征...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 训练OLS模型
print("\n训练OLS模型...")
ols = LinearRegression()
ols.fit(X_train_scaled, y_train_lnprice)

# 预测
print("进行预测...")
y_train_pred_ln = ols.predict(X_train_scaled)
y_test_pred_ln = ols.predict(X_test_scaled)

# 转换回原始价格
y_train_pred = np.exp(y_train_pred_ln)
y_test_pred = np.exp(y_test_pred_ln)
y_train_true_price = np.exp(y_train_lnprice)
y_test_true_price = np.exp(y_test_lnprice)

# 计算性能指标
print("\n计算性能指标...")
train_mae = mean_absolute_error(y_train_true_price, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train_true_price, y_train_pred))
train_r2 = r2_score(y_train_true_price, y_train_pred)

test_mae = mean_absolute_error(y_test_true_price, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_true_price, y_test_pred))
test_r2 = r2_score(y_test_true_price, y_test_pred)

# 6折交叉验证
print("进行6折交叉验证...")
kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_mae_scores = -cross_val_score(ols, X_train_scaled, y_train_price, 
                                cv=kf, scoring='neg_mean_absolute_error')
cv_rmse_scores = np.sqrt(-cross_val_score(ols, X_train_scaled, y_train_price, 
                                         cv=kf, scoring='neg_mean_squared_error'))

cv_mae = np.mean(cv_mae_scores)
cv_rmse = np.mean(cv_rmse_scores)

# 输出结果
print("\n" + "=" * 80)
print("OLS模型性能汇总")
print("=" * 80)

print(f"{'指标':<20} {'训练集':<15} {'测试集':<15} {'交叉验证':<15}")
print("-" * 65)
print(f"{'MAE':<20} {train_mae:<15.2f} {test_mae:<15.2f} {cv_mae:<15.2f}")
print(f"{'RMSE':<20} {train_rmse:<15.2f} {test_rmse:<15.2f} {cv_rmse:<15.2f}")
print(f"{'R²':<20} {train_r2:<15.4f} {test_r2:<15.4f} {'-':<15}")

print(f"\n总预测数量: {len(X)}")
print("✅ OLS模型训练完成")

# 存储模型和预处理对象（不保存到文件，只存储在内存中）
print("\n模型和预处理对象已存储在内存中")


OLS 模型建模
读取训练集数据...
训练集形状: (98899, 82)

准备特征变量和目标变量...
特征数量: 80
样本数量: 98899

处理缺失值...
  - 区县: 4677 个缺失值已用中位数填充
  - 板块: 5144 个缺失值已用中位数填充

划分训练集和测试集...
训练集大小: (79119, 80)
测试集大小: (19780, 80)

标准化特征...

训练OLS模型...
进行预测...

计算性能指标...
进行6折交叉验证...

OLS模型性能汇总
指标                   训练集             测试集             交叉验证           
-----------------------------------------------------------------
MAE                  152363.89       153361.48       218644.75      
RMSE                 339809.73       352232.41       378487.06      
R²                   0.6960          0.7005          -              

总预测数量: 98899
✅ OLS模型训练完成

模型和预处理对象已存储在内存中


In [18]:
# 2.2 OLS 模型预测
print("\n" + "=" * 60)
print("OLS 模型预测")
print("=" * 60)

# 读取测试集数据
print("读取测试集数据...")
test_df = pd.read_csv('rent_price_test_dataset.csv')
print(f"测试集形状: {test_df.shape}")

print(f"使用 {len(feature_cols)} 个特征进行预测")

# 确保测试集包含所有需要的特征
print("\n确保特征一致性...")
missing_features = set(feature_cols) - set(test_df.columns)
if missing_features:
    print(f"创建缺失的特征并设为0: {len(missing_features)} 个")
    for feature in missing_features:
        test_df[feature] = 0
        print(f"  - 已创建: {feature}")

# 选择特征
X_test = test_df[feature_cols]

# 处理缺失值
print("\n处理缺失值...")
for col in X_test.columns:
    if X_test[col].isnull().sum() > 0:
        null_count = X_test[col].isnull().sum()
        if X_test[col].dtype in ['float64', 'int64']:
            median_val = X_test[col].median()
            X_test[col] = X_test[col].fillna(median_val)
            print(f"  - {col}: {null_count} 个缺失值已用中位数填充")
        else:
            mode_val = X_test[col].mode()[0] if not X_test[col].mode().empty else 0
            X_test[col] = X_test[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已用众数填充")

# 应用标准化
print("\n应用标准化...")
X_test_scaled = scaler.transform(X_test)

# 进行预测
print("进行预测...")
y_test_pred_ln = ols.predict(X_test_scaled)

# 将对数价格转换回原始价格
y_test_pred = np.exp(y_test_pred_ln)

# 创建结果DataFrame
if 'ID' in test_df.columns:
    result_df = pd.DataFrame({
        'ID': test_df['ID'],
        'Price': y_test_pred
    })
else:
    result_df = pd.DataFrame({
        'ID': range(len(y_test_pred)),
        'Price': y_test_pred
    })

# 保存预测结果
output_path = 'rent_price_ols_predictions.csv'
result_df.to_csv(output_path, index=False, float_format='%.2f')
print(f"\n预测结果已保存到: {output_path}")
print(f"结果文件包含 {len(result_df)} 条预测记录")

# 显示预测结果的统计信息
print("\n预测结果统计信息:")
print(f"预测值 Price 范围: [{result_df['Price'].min():.2f}, {result_df['Price'].max():.2f}]")
print(f"预测值 Price 均值: {result_df['Price'].mean():.2f}")
print(f"预测值 Price 标准差: {result_df['Price'].std():.2f}")

print("\n✅ OLS模型预测完成")


OLS 模型预测
读取测试集数据...
测试集形状: (9773, 81)
使用 80 个特征进行预测

确保特征一致性...

处理缺失值...
  - 区县: 925 个缺失值已用中位数填充
  - 板块: 936 个缺失值已用中位数填充

应用标准化...
进行预测...

预测结果已保存到: rent_price_ols_predictions.csv
结果文件包含 9773 条预测记录

预测结果统计信息:
预测值 Price 范围: [56697.04, 9944285.95]
预测值 Price 均值: 507833.32
预测值 Price 标准差: 458564.82

✅ OLS模型预测完成


In [20]:
# 3.1 LASSO模型建模
print("\n" + "=" * 60)
print("LASSO 模型建模")
print("=" * 60)

from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 读取训练集数据
print("读取训练集数据...")
df = pd.read_csv('rent_price_train_dataset.csv')
print(f"训练集形状: {df.shape}")

# 准备特征变量和目标变量
print("\n准备特征变量和目标变量...")

# 选择特征 - 排除目标变量和ID等
feature_cols = [col for col in df.columns if col not in ['Price', 'lnPrice', 'ID'] 
                and df[col].dtype in ['int64', 'float64']]

X = df[feature_cols]
y_price = df['Price']  # 原始价格
y_lnprice = df['lnPrice']  # 对数价格

print(f"特征数量: {len(feature_cols)}")
print(f"样本数量: {len(X)}")

# 处理缺失值
print("\n处理缺失值...")
for col in X.columns:
    if X[col].isnull().sum() > 0:
        null_count = X[col].isnull().sum()
        if X[col].dtype in ['float64', 'int64']:
            median_val = X[col].median()
            X[col] = X[col].fillna(median_val)
            print(f"  - {col}: {null_count} 个缺失值已用中位数填充")
        else:
            mode_val = X[col].mode()[0] if not X[col].mode().empty else 0
            X[col] = X[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已用众数填充")

# 划分训练集和测试集
print("\n划分训练集和测试集...")
X_train, X_test, y_train_price, y_test_price = train_test_split(
    X, y_price, test_size=0.2, random_state=42
)
_, _, y_train_lnprice, y_test_lnprice = train_test_split(
    X, y_lnprice, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 标准化特征
print("\n标准化特征...")
lasso_scaler = StandardScaler()
X_train_scaled = lasso_scaler.fit_transform(X_train)
X_test_scaled = lasso_scaler.transform(X_test)

# 定义参数网格
lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}

# 使用GridSearchCV进行超参数调优
print("\n使用GridSearchCV进行LASSO超参数调优...")
lasso = Lasso(random_state=42)
lasso_grid = GridSearchCV(lasso, lasso_params, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
lasso_grid.fit(X_train_scaled, y_train_lnprice)

# 获取最佳模型
best_lasso = lasso_grid.best_estimator_
print(f"最佳 alpha: {lasso_grid.best_params_['alpha']}")

# 预测
print("进行预测...")
y_train_pred_ln = best_lasso.predict(X_train_scaled)
y_test_pred_ln = best_lasso.predict(X_test_scaled)

# 转换回原始价格
y_train_pred = np.exp(y_train_pred_ln)
y_test_pred = np.exp(y_test_pred_ln)
y_train_true_price = np.exp(y_train_lnprice)
y_test_true_price = np.exp(y_test_lnprice)

# 计算性能指标
print("\n计算性能指标...")
train_mae = mean_absolute_error(y_train_true_price, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train_true_price, y_train_pred))
train_r2 = r2_score(y_train_true_price, y_train_pred)

test_mae = mean_absolute_error(y_test_true_price, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_true_price, y_test_pred))
test_r2 = r2_score(y_test_true_price, y_test_pred)

# 6折交叉验证
print("进行6折交叉验证...")
kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_mae_scores = -cross_val_score(best_lasso, X_train_scaled, y_train_price, 
                                cv=kf, scoring='neg_mean_absolute_error')
cv_rmse_scores = np.sqrt(-cross_val_score(best_lasso, X_train_scaled, y_train_price, 
                                         cv=kf, scoring='neg_mean_squared_error'))

cv_mae = np.mean(cv_mae_scores)
cv_rmse = np.mean(cv_rmse_scores)

# 输出结果
print("\n" + "=" * 80)
print("LASSO模型性能汇总")
print("=" * 80)

print(f"{'指标':<20} {'训练集':<15} {'测试集':<15} {'交叉验证':<15}")
print("-" * 65)
print(f"{'MAE':<20} {train_mae:<15.2f} {test_mae:<15.2f} {cv_mae:<15.2f}")
print(f"{'RMSE':<20} {train_rmse:<15.2f} {test_rmse:<15.2f} {cv_rmse:<15.2f}")
print(f"{'R²':<20} {train_r2:<15.4f} {test_r2:<15.4f} {'-':<15}")

print(f"\n总预测数量: {len(X)}")
print("✅ LASSO模型训练完成")

# 显示特征重要性
print("\nLASSO模型特征重要性 (前10个):")
lasso_coef = pd.DataFrame({
    '特征': feature_cols,
    '系数': best_lasso.coef_
}).sort_values('系数', key=abs, ascending=False)

print(lasso_coef.head(10))

# 统计被压缩为0的特征数量
zero_coef_count = (best_lasso.coef_ == 0).sum()
print(f"\n被LASSO压缩为0的特征数量: {zero_coef_count}/{len(feature_cols)}")


LASSO 模型建模
读取训练集数据...
训练集形状: (98899, 82)

准备特征变量和目标变量...
特征数量: 80
样本数量: 98899

处理缺失值...
  - 区县: 4677 个缺失值已用中位数填充
  - 板块: 5144 个缺失值已用中位数填充

划分训练集和测试集...
训练集大小: (79119, 80)
测试集大小: (19780, 80)

标准化特征...

使用GridSearchCV进行LASSO超参数调优...
最佳 alpha: 0.001
进行预测...

计算性能指标...
进行6折交叉验证...

LASSO模型性能汇总
指标                   训练集             测试集             交叉验证           
-----------------------------------------------------------------
MAE                  154449.17       155459.78       218635.88      
RMSE                 344991.27       358575.97       378530.80      
R²                   0.6866          0.6896          -              

总预测数量: 98899
✅ LASSO模型训练完成

LASSO模型特征重要性 (前10个):
              特征        系数
63        city_7  0.620179
66       city_10  0.442084
78  city_dist_10 -0.442003
75   city_dist_7 -0.440527
1           area  0.340815
72   city_dist_4  0.290680
56        city_0  0.279292
61        city_5 -0.185631
57        city_1 -0.153181
73   city_dist_5  0.138788

被LASSO压缩为0的特征数量: 1

In [21]:
# 3.2 LASSO 模型预测
print("\n" + "=" * 60)
print("LASSO 模型预测")
print("=" * 60)

# 读取测试集数据
print("读取测试集数据...")
test_df = pd.read_csv('rent_price_test_dataset.csv')
print(f"测试集形状: {test_df.shape}")

print(f"使用 {len(feature_cols)} 个特征进行预测")

# 确保测试集包含所有需要的特征
print("\n确保特征一致性...")
missing_features = set(feature_cols) - set(test_df.columns)
if missing_features:
    print(f"创建缺失的特征并设为0: {len(missing_features)} 个")
    for feature in missing_features:
        test_df[feature] = 0
        print(f"  - 已创建: {feature}")

# 选择特征
X_test = test_df[feature_cols]

# 处理缺失值
print("\n处理缺失值...")
for col in X_test.columns:
    if X_test[col].isnull().sum() > 0:
        null_count = X_test[col].isnull().sum()
        if X_test[col].dtype in ['float64', 'int64']:
            median_val = X_test[col].median()
            X_test[col] = X_test[col].fillna(median_val)
            print(f"  - {col}: {null_count} 个缺失值已用中位数填充")
        else:
            mode_val = X_test[col].mode()[0] if not X_test[col].mode().empty else 0
            X_test[col] = X_test[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已用众数填充")

# 应用标准化
print("\n应用标准化...")
X_test_scaled = lasso_scaler.transform(X_test)

# 进行预测
print("进行预测...")
y_test_pred_ln = best_lasso.predict(X_test_scaled)

# 将对数价格转换回原始价格
y_test_pred = np.exp(y_test_pred_ln)

# 创建结果DataFrame
if 'ID' in test_df.columns:
    result_df = pd.DataFrame({
        'ID': test_df['ID'],
        'Price': y_test_pred
    })
else:
    result_df = pd.DataFrame({
        'ID': range(len(y_test_pred)),
        'Price': y_test_pred
    })

# 保存预测结果
output_path = 'rent_price_lasso_predictions.csv'
result_df.to_csv(output_path, index=False, float_format='%.2f')
print(f"\n预测结果已保存到: {output_path}")
print(f"结果文件包含 {len(result_df)} 条预测记录")

# 显示预测结果的统计信息
print("\n预测结果统计信息:")
print(f"预测值 Price 范围: [{result_df['Price'].min():.2f}, {result_df['Price'].max():.2f}]")
print(f"预测值 Price 均值: {result_df['Price'].mean():.2f}")
print(f"预测值 Price 标准差: {result_df['Price'].std():.2f}")

print("\n✅ LASSO模型预测完成")


LASSO 模型预测
读取测试集数据...
测试集形状: (9773, 81)
使用 80 个特征进行预测

确保特征一致性...

处理缺失值...
  - 区县: 925 个缺失值已用中位数填充
  - 板块: 936 个缺失值已用中位数填充

应用标准化...
进行预测...

预测结果已保存到: rent_price_lasso_predictions.csv
结果文件包含 9773 条预测记录

预测结果统计信息:
预测值 Price 范围: [59739.74, 10261011.17]
预测值 Price 均值: 512380.75
预测值 Price 标准差: 459698.62

✅ LASSO模型预测完成


In [23]:
# 4.1 Ridge 模型建模
print("\n" + "=" * 60)
print("Ridge 模型建模")
print("=" * 60)

from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 读取训练集数据
print("读取训练集数据...")
df = pd.read_csv('rent_price_train_dataset.csv')
print(f"训练集形状: {df.shape}")

# 准备特征变量和目标变量
print("\n准备特征变量和目标变量...")

# 选择特征 - 排除目标变量和ID等
feature_cols = [col for col in df.columns if col not in ['Price', 'lnPrice', 'ID'] 
                and df[col].dtype in ['int64', 'float64']]

X = df[feature_cols]
y_price = df['Price']  # 原始价格
y_lnprice = df['lnPrice']  # 对数价格

print(f"特征数量: {len(feature_cols)}")
print(f"样本数量: {len(X)}")

# 处理缺失值
print("\n处理缺失值...")
for col in X.columns:
    if X[col].isnull().sum() > 0:
        null_count = X[col].isnull().sum()
        if X[col].dtype in ['float64', 'int64']:
            median_val = X[col].median()
            X[col] = X[col].fillna(median_val)
            print(f"  - {col}: {null_count} 个缺失值已用中位数填充")
        else:
            mode_val = X[col].mode()[0] if not X[col].mode().empty else 0
            X[col] = X[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已用众数填充")

# 划分训练集和测试集
print("\n划分训练集和测试集...")
X_train, X_test, y_train_price, y_test_price = train_test_split(
    X, y_price, test_size=0.2, random_state=42
)
_, _, y_train_lnprice, y_test_lnprice = train_test_split(
    X, y_lnprice, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 标准化特征
print("\n标准化特征...")
ridge_scaler = StandardScaler()
X_train_scaled = ridge_scaler.fit_transform(X_train)
X_test_scaled = ridge_scaler.transform(X_test)

# 定义参数网格
ridge_params = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

# 使用GridSearchCV进行超参数调优
print("\n使用GridSearchCV进行Ridge超参数调优...")
ridge = Ridge(random_state=42)
ridge_grid = GridSearchCV(ridge, ridge_params, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_grid.fit(X_train_scaled, y_train_lnprice)

# 获取最佳模型
best_ridge = ridge_grid.best_estimator_
print(f"最佳 alpha: {ridge_grid.best_params_['alpha']}")

# 预测
print("进行预测...")
y_train_pred_ln = best_ridge.predict(X_train_scaled)
y_test_pred_ln = best_ridge.predict(X_test_scaled)

# 转换回原始价格
y_train_pred = np.exp(y_train_pred_ln)
y_test_pred = np.exp(y_test_pred_ln)
y_train_true_price = np.exp(y_train_lnprice)
y_test_true_price = np.exp(y_test_lnprice)

# 计算性能指标
print("\n计算性能指标...")
train_mae = mean_absolute_error(y_train_true_price, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train_true_price, y_train_pred))
train_r2 = r2_score(y_train_true_price, y_train_pred)

test_mae = mean_absolute_error(y_test_true_price, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_true_price, y_test_pred))
test_r2 = r2_score(y_test_true_price, y_test_pred)

# 6折交叉验证
print("进行6折交叉验证...")
kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_mae_scores = -cross_val_score(best_ridge, X_train_scaled, y_train_price, 
                                cv=kf, scoring='neg_mean_absolute_error')
cv_rmse_scores = np.sqrt(-cross_val_score(best_ridge, X_train_scaled, y_train_price, 
                                         cv=kf, scoring='neg_mean_squared_error'))

cv_mae = np.mean(cv_mae_scores)
cv_rmse = np.mean(cv_rmse_scores)

# 输出结果
print("\n" + "=" * 80)
print("Ridge模型性能汇总")
print("=" * 80)

print(f"{'指标':<20} {'训练集':<15} {'测试集':<15} {'交叉验证':<15}")
print("-" * 65)
print(f"{'MAE':<20} {train_mae:<15.2f} {test_mae:<15.2f} {cv_mae:<15.2f}")
print(f"{'RMSE':<20} {train_rmse:<15.2f} {test_rmse:<15.2f} {cv_rmse:<15.2f}")
print(f"{'R²':<20} {train_r2:<15.4f} {test_r2:<15.4f} {'-':<15}")

print(f"\n总预测数量: {len(X)}")
print("✅ Ridge模型训练完成")

# 显示特征重要性
print("\nRidge模型特征重要性 (前10个):")
ridge_coef = pd.DataFrame({
    '特征': feature_cols,
    '系数': best_ridge.coef_
}).sort_values('系数', key=abs, ascending=False)

print(ridge_coef.head(10))

# 显示系数分布
print(f"\nRidge模型系数统计:")
print(f"系数绝对值均值: {np.abs(best_ridge.coef_).mean():.4f}")
print(f"系数绝对值标准差: {np.abs(best_ridge.coef_).std():.4f}")
print(f"最大系数: {best_ridge.coef_.max():.4f}")
print(f"最小系数: {best_ridge.coef_.min():.4f}")


Ridge 模型建模
读取训练集数据...
训练集形状: (98899, 82)

准备特征变量和目标变量...
特征数量: 80
样本数量: 98899

处理缺失值...
  - 区县: 4677 个缺失值已用中位数填充
  - 板块: 5144 个缺失值已用中位数填充

划分训练集和测试集...
训练集大小: (79119, 80)
测试集大小: (19780, 80)

标准化特征...

使用GridSearchCV进行Ridge超参数调优...
最佳 alpha: 1
进行预测...

计算性能指标...
进行6折交叉验证...

Ridge模型性能汇总
指标                   训练集             测试集             交叉验证           
-----------------------------------------------------------------
MAE                  152366.80       153360.71       218628.66      
RMSE                 339840.43       352279.47       378486.88      
R²                   0.6959          0.7004          -              

总预测数量: 98899
✅ Ridge模型训练完成

Ridge模型特征重要性 (前10个):
              特征        系数
75   city_dist_7 -0.816603
63        city_7  0.787555
78  city_dist_10 -0.697784
61        city_5 -0.538584
77   city_dist_9 -0.480044
58        city_2 -0.468637
48            城市  0.419617
73   city_dist_5  0.404480
56        city_0  0.391386
72   city_dist_4  0.375268

Ridge模型系数统计:
系数绝对值均值: 

In [24]:
# 4.2 Ridge 模型预测
print("\n" + "=" * 60)
print("Ridge 模型预测")
print("=" * 60)

# 读取测试集数据
print("读取测试集数据...")
test_df = pd.read_csv('rent_price_test_dataset.csv')
print(f"测试集形状: {test_df.shape}")

print(f"使用 {len(feature_cols)} 个特征进行预测")

# 确保测试集包含所有需要的特征
print("\n确保特征一致性...")
missing_features = set(feature_cols) - set(test_df.columns)
if missing_features:
    print(f"创建缺失的特征并设为0: {len(missing_features)} 个")
    for feature in missing_features:
        test_df[feature] = 0
        print(f"  - 已创建: {feature}")

# 选择特征
X_test = test_df[feature_cols]

# 处理缺失值
print("\n处理缺失值...")
for col in X_test.columns:
    if X_test[col].isnull().sum() > 0:
        null_count = X_test[col].isnull().sum()
        if X_test[col].dtype in ['float64', 'int64']:
            median_val = X_test[col].median()
            X_test[col] = X_test[col].fillna(median_val)
            print(f"  - {col}: {null_count} 个缺失值已用中位数填充")
        else:
            mode_val = X_test[col].mode()[0] if not X_test[col].mode().empty else 0
            X_test[col] = X_test[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已用众数填充")

# 应用标准化
print("\n应用标准化...")
X_test_scaled = ridge_scaler.transform(X_test)

# 进行预测
print("进行预测...")
y_test_pred_ln = best_ridge.predict(X_test_scaled)

# 将对数价格转换回原始价格
y_test_pred = np.exp(y_test_pred_ln)

# 创建结果DataFrame
if 'ID' in test_df.columns:
    result_df = pd.DataFrame({
        'ID': test_df['ID'],
        'Price': y_test_pred
    })
else:
    result_df = pd.DataFrame({
        'ID': range(len(y_test_pred)),
        'Price': y_test_pred
    })

# 保存预测结果
output_path = 'rent_price_ridge_predictions.csv'
result_df.to_csv(output_path, index=False, float_format='%.2f')
print(f"\n预测结果已保存到: {output_path}")
print(f"结果文件包含 {len(result_df)} 条预测记录")

# 显示预测结果的统计信息
print("\n预测结果统计信息:")
print(f"预测值 Price 范围: [{result_df['Price'].min():.2f}, {result_df['Price'].max():.2f}]")
print(f"预测值 Price 均值: {result_df['Price'].mean():.2f}")
print(f"预测值 Price 标准差: {result_df['Price'].std():.2f}")

print("\n✅ Ridge模型预测完成")


Ridge 模型预测
读取测试集数据...
测试集形状: (9773, 81)
使用 80 个特征进行预测

确保特征一致性...

处理缺失值...
  - 区县: 925 个缺失值已用中位数填充
  - 板块: 936 个缺失值已用中位数填充

应用标准化...
进行预测...

预测结果已保存到: rent_price_ridge_predictions.csv
结果文件包含 9773 条预测记录

预测结果统计信息:
预测值 Price 范围: [56708.17, 9947080.31]
预测值 Price 均值: 507818.10
预测值 Price 标准差: 458552.48

✅ Ridge模型预测完成


In [25]:
# 5.1 Elastic Net 模型建模
print("\n" + "=" * 60)
print("Elastic Net 模型建模")
print("=" * 60)

from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 读取训练集数据
print("读取训练集数据...")
df = pd.read_csv('rent_price_train_dataset.csv')
print(f"训练集形状: {df.shape}")

# 准备特征变量和目标变量
print("\n准备特征变量和目标变量...")

# 选择特征 - 排除目标变量和ID等
feature_cols = [col for col in df.columns if col not in ['Price', 'lnPrice', 'ID'] 
                and df[col].dtype in ['int64', 'float64']]

X = df[feature_cols]
y_price = df['Price']  # 原始价格
y_lnprice = df['lnPrice']  # 对数价格

print(f"特征数量: {len(feature_cols)}")
print(f"样本数量: {len(X)}")

# 处理缺失值
print("\n处理缺失值...")
for col in X.columns:
    if X[col].isnull().sum() > 0:
        null_count = X[col].isnull().sum()
        if X[col].dtype in ['float64', 'int64']:
            median_val = X[col].median()
            X[col] = X[col].fillna(median_val)
            print(f"  - {col}: {null_count} 个缺失值已用中位数填充")
        else:
            mode_val = X[col].mode()[0] if not X[col].mode().empty else 0
            X[col] = X[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已用众数填充")

# 划分训练集和测试集
print("\n划分训练集和测试集...")
X_train, X_test, y_train_price, y_test_price = train_test_split(
    X, y_price, test_size=0.2, random_state=42
)
_, _, y_train_lnprice, y_test_lnprice = train_test_split(
    X, y_lnprice, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 标准化特征
print("\n标准化特征...")
elastic_scaler = StandardScaler()
X_train_scaled = elastic_scaler.fit_transform(X_train)
X_test_scaled = elastic_scaler.transform(X_test)

# 定义参数网格
elastic_params = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

# 使用GridSearchCV进行超参数调优
print("\n使用GridSearchCV进行Elastic Net超参数调优...")
elastic = ElasticNet(random_state=42, max_iter=10000)
elastic_grid = GridSearchCV(elastic, elastic_params, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
elastic_grid.fit(X_train_scaled, y_train_lnprice)

# 获取最佳模型
best_elastic = elastic_grid.best_estimator_
print(f"最佳参数: alpha={elastic_grid.best_params_['alpha']}, l1_ratio={elastic_grid.best_params_['l1_ratio']}")

# 预测
print("进行预测...")
y_train_pred_ln = best_elastic.predict(X_train_scaled)
y_test_pred_ln = best_elastic.predict(X_test_scaled)

# 转换回原始价格
y_train_pred = np.exp(y_train_pred_ln)
y_test_pred = np.exp(y_test_pred_ln)
y_train_true_price = np.exp(y_train_lnprice)
y_test_true_price = np.exp(y_test_lnprice)

# 计算性能指标
print("\n计算性能指标...")
train_mae = mean_absolute_error(y_train_true_price, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train_true_price, y_train_pred))
train_r2 = r2_score(y_train_true_price, y_train_pred)

test_mae = mean_absolute_error(y_test_true_price, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_true_price, y_test_pred))
test_r2 = r2_score(y_test_true_price, y_test_pred)

# 6折交叉验证
print("进行6折交叉验证...")
kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_mae_scores = -cross_val_score(best_elastic, X_train_scaled, y_train_price, 
                                cv=kf, scoring='neg_mean_absolute_error')
cv_rmse_scores = np.sqrt(-cross_val_score(best_elastic, X_train_scaled, y_train_price, 
                                         cv=kf, scoring='neg_mean_squared_error'))

cv_mae = np.mean(cv_mae_scores)
cv_rmse = np.mean(cv_rmse_scores)

# 输出结果
print("\n" + "=" * 80)
print("Elastic Net模型性能汇总")
print("=" * 80)

print(f"{'指标':<20} {'训练集':<15} {'测试集':<15} {'交叉验证':<15}")
print("-" * 65)
print(f"{'MAE':<20} {train_mae:<15.2f} {test_mae:<15.2f} {cv_mae:<15.2f}")
print(f"{'RMSE':<20} {train_rmse:<15.2f} {test_rmse:<15.2f} {cv_rmse:<15.2f}")
print(f"{'R²':<20} {train_r2:<15.4f} {test_r2:<15.4f} {'-':<15}")

print(f"\n总预测数量: {len(X)}")
print("✅ Elastic Net模型训练完成")

# 显示特征重要性
print("\nElastic Net模型特征重要性 (前10个):")
elastic_coef = pd.DataFrame({
    '特征': feature_cols,
    '系数': best_elastic.coef_
}).sort_values('系数', key=abs, ascending=False)

print(elastic_coef.head(10))

# 统计被压缩为0的特征数量
zero_coef_count = (best_elastic.coef_ == 0).sum()
print(f"\n被Elastic Net压缩为0的特征数量: {zero_coef_count}/{len(feature_cols)}")

# 显示正则化类型
l1_ratio = elastic_grid.best_params_['l1_ratio']
if l1_ratio == 1:
    print("模型行为类似于LASSO (纯L1正则化)")
elif l1_ratio == 0:
    print("模型行为类似于Ridge (纯L2正则化)")
else:
    print(f"模型使用混合正则化 (L1比例: {l1_ratio})")


Elastic Net 模型建模
读取训练集数据...
训练集形状: (98899, 82)

准备特征变量和目标变量...
特征数量: 80
样本数量: 98899

处理缺失值...
  - 区县: 4677 个缺失值已用中位数填充
  - 板块: 5144 个缺失值已用中位数填充

划分训练集和测试集...
训练集大小: (79119, 80)
测试集大小: (19780, 80)

标准化特征...

使用GridSearchCV进行Elastic Net超参数调优...
最佳参数: alpha=0.001, l1_ratio=0.1
进行预测...

计算性能指标...
进行6折交叉验证...

Elastic Net模型性能汇总
指标                   训练集             测试集             交叉验证           
-----------------------------------------------------------------
MAE                  153158.91       154086.21       218482.46      
RMSE                 342434.42       355601.61       378755.85      
R²                   0.6912          0.6948          -              

总预测数量: 98899
✅ Elastic Net模型训练完成

Elastic Net模型特征重要性 (前10个):
              特征        系数
63        city_7  0.617898
75   city_dist_7 -0.569345
78  city_dist_10 -0.558475
61        city_5 -0.404159
66       city_10  0.340763
1           area  0.340288
58        city_2 -0.303537
73   city_dist_5  0.299691
72   city_dist_4  0.289206


In [27]:
# 5.2 Elastic Net 模型预测
print("\n" + "=" * 60)
print("Elastic Net 模型预测")
print("=" * 60)

# 读取测试集数据
print("读取测试集数据...")
test_df = pd.read_csv('rent_price_test_dataset.csv')
print(f"测试集形状: {test_df.shape}")

print(f"使用 {len(feature_cols)} 个特征进行预测")

# 确保测试集包含所有需要的特征
print("\n确保特征一致性...")
missing_features = set(feature_cols) - set(test_df.columns)
if missing_features:
    print(f"创建缺失的特征并设为0: {len(missing_features)} 个")
    for feature in missing_features:
        test_df[feature] = 0
        print(f"  - 已创建: {feature}")

# 选择特征
X_test = test_df[feature_cols]

# 处理缺失值
print("\n处理缺失值...")
for col in X_test.columns:
    if X_test[col].isnull().sum() > 0:
        null_count = X_test[col].isnull().sum()
        if X_test[col].dtype in ['float64', 'int64']:
            median_val = X_test[col].median()
            X_test[col] = X_test[col].fillna(median_val)
            print(f"  - {col}: {null_count} 个缺失值已用中位数填充")
        else:
            mode_val = X_test[col].mode()[0] if not X_test[col].mode().empty else 0
            X_test[col] = X_test[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已用众数填充")

# 应用标准化
print("\n应用标准化...")
X_test_scaled = elastic_scaler.transform(X_test)

# 进行预测
print("进行预测...")
y_test_pred_ln = best_elastic.predict(X_test_scaled)

# 将对数价格转换回原始价格
y_test_pred = np.exp(y_test_pred_ln)

# 创建结果DataFrame
if 'ID' in test_df.columns:
    result_df = pd.DataFrame({
        'ID': test_df['ID'],
        'Price': y_test_pred
    })
else:
    result_df = pd.DataFrame({
        'ID': range(len(y_test_pred)),
        'Price': y_test_pred
    })

# 保存预测结果
output_path = 'rent_price_elastic_net_predictions.csv'
result_df.to_csv(output_path, index=False, float_format='%.2f')
print(f"\n预测结果已保存到: {output_path}")
print(f"结果文件包含 {len(result_df)} 条预测记录")

# 显示预测结果的统计信息
print("\n预测结果统计信息:")
print(f"预测值 Price 范围: [{result_df['Price'].min():.2f}, {result_df['Price'].max():.2f}]")
print(f"预测值 Price 均值: {result_df['Price'].mean():.2f}")
print(f"预测值 Price 标准差: {result_df['Price'].std():.2f}")

print("\n✅ Elastic Net模型预测完成")


Elastic Net 模型预测
读取测试集数据...
测试集形状: (9773, 81)
使用 80 个特征进行预测

确保特征一致性...

处理缺失值...
  - 区县: 925 个缺失值已用中位数填充
  - 板块: 936 个缺失值已用中位数填充

应用标准化...
进行预测...

预测结果已保存到: rent_price_elastic_net_predictions.csv
结果文件包含 9773 条预测记录

预测结果统计信息:
预测值 Price 范围: [58311.22, 10059491.08]
预测值 Price 均值: 507603.04
预测值 Price 标准差: 457287.99

✅ Elastic Net模型预测完成
